<a href="https://colab.research.google.com/github/SBethune103/virtual-running-coach-pipeline/blob/dev/notebooks/02_data_wrangling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q polars pyarrow

import polars as pl
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Paths point to Google Drive
BASE = Path("/content/drive/MyDrive/virtual-running-coach")
DATA_RAW = BASE / "data/raw"
DATA_PROCESSED = BASE / "data/processed"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("✅ Setup complete")
print("Raw data location:", DATA_RAW)
print("Processed data location:", DATA_PROCESSED)

Mounted at /content/drive
✅ Setup complete
Raw data location: /content/drive/MyDrive/virtual-running-coach/data/raw
Processed data location: /content/drive/MyDrive/virtual-running-coach/data/processed


In [ ]:
# Find weekly files
weekly_files = list(DATA_RAW.glob("run_ww_*_w.csv"))
print(f"Found {len(weekly_files)} weekly files")

if len(weekly_files) == 0:
    print("❌ No weekly files found. Please re-run the download cell in Notebook 1.")
else:
    # Load a sample (start with the first weekly file)
    df_weekly = pl.read_csv(
        weekly_files[0],
        try_parse_dates=True,
        null_values=["", "NA"]
    )

    print(f"Loaded: {df_weekly.shape[0]:,} rows × {df_weekly.shape[1]} columns")
    print(df_weekly.head())
    print("\nColumn types:")
    print(df_weekly.schema)

Found 2 weekly files
Loaded: 1,893,424 rows × 9 columns
shape: (5, 9)
┌─────┬────────────┬─────────┬──────────┬───┬────────┬───────────┬────────────────┬────────────────┐
│     ┆ datetime   ┆ athlete ┆ distance ┆ … ┆ gender ┆ age_group ┆ country        ┆ major          │
│ --- ┆ ---        ┆ ---     ┆ ---      ┆   ┆ ---    ┆ ---       ┆ ---            ┆ ---            │
│ i64 ┆ date       ┆ i64     ┆ f64      ┆   ┆ str    ┆ str       ┆ str            ┆ str            │
╞═════╪════════════╪═════════╪══════════╪═══╪════════╪═══════════╪════════════════╪════════════════╡
│ 0   ┆ 2019-01-01 ┆ 0       ┆ 0.0      ┆ … ┆ F      ┆ 18 - 34   ┆ United States  ┆ CHICAGO 2019   │
│ 1   ┆ 2019-01-01 ┆ 1       ┆ 5.27     ┆ … ┆ M      ┆ 35 - 54   ┆ Germany        ┆ BERLIN 2016    │
│ 2   ┆ 2019-01-01 ┆ 2       ┆ 9.3      ┆ … ┆ M      ┆ 35 - 54   ┆ United Kingdom ┆ LONDON         │
│     ┆            ┆         ┆          ┆   ┆        ┆           ┆                ┆ 2018,LONDON    │
│     ┆            ┆ 

In [ ]:
def clean_running_data(df: pl.DataFrame) -> pl.DataFrame:
    """Clean and engineer features for long-distance running data"""

    # First make sure we have a proper date column
    if df["datetime"].dtype == pl.Utf8:  # still a string
        df = df.with_columns(
            pl.col("datetime").str.to_datetime(strict=False).alias("date")
        )
    else:
        # already a date/datetime type
        df = df.with_columns(
            pl.col("datetime").alias("date")
        )

    df = (
        df
        # Filter out zero-distance activities
        .filter(pl.col("distance") > 0)

        # Create useful features
        .with_columns([
            (pl.col("distance") / (pl.col("duration") / 60)).alias("avg_speed_kmh"),  # km/h
            (pl.col("duration") / pl.col("distance")).alias("pace_min_per_km"),       # min/km
            pl.col("date").dt.year().alias("year"),
            pl.col("date").dt.month().alias("month"),
            pl.col("date").dt.weekday().alias("weekday"),  # 1=Mon ... 7=Sun
        ])

        # Clean up
        .drop_nulls(subset=["distance", "duration"])
    )

    return df


df_clean = clean_running_data(df_weekly)

print(f"After cleaning: {df_clean.shape[0]:,} rows")
print(df_clean.select(["distance", "duration", "avg_speed_kmh", "pace_min_per_km"]).describe())

After cleaning: 1,425,022 rows
shape: (9, 5)
┌────────────┬────────────┬────────────┬───────────────┬─────────────────┐
│ statistic  ┆ distance   ┆ duration   ┆ avg_speed_kmh ┆ pace_min_per_km │
│ ---        ┆ ---        ┆ ---        ┆ ---           ┆ ---             │
│ str        ┆ f64        ┆ f64        ┆ f64           ┆ f64             │
╞════════════╪════════════╪════════════╪═══════════════╪═════════════════╡
│ count      ┆ 1.425022e6 ┆ 1.425022e6 ┆ 1.425022e6    ┆ 1.425022e6      │
│ null_count ┆ 0.0        ┆ 0.0        ┆ 0.0           ┆ 0.0             │
│ mean       ┆ 38.853018  ┆ 213.364897 ┆ 10.928763     ┆ 5.856122        │
│ std        ┆ 28.823443  ┆ 159.942917 ┆ 1.86155       ┆ 74.816444       │
│ min        ┆ 0.00875    ┆ 0.016667   ┆ 0.000682      ┆ 1.666667        │
│ 25%        ┆ 16.38      ┆ 93.85      ┆ 9.870588      ┆ 4.926199        │
│ 50%        ┆ 32.68      ┆ 182.95     ┆ 11.051954     ┆ 5.428905        │
│ 75%        ┆ 54.24      ┆ 296.633333 ┆ 12.179775     

In [ ]:
print("=== Quick Insights ===")
print(f"\nTotal athletes: {df_clean['athlete'].n_unique():,}")
print(f"Average weekly distance: {df_clean['distance'].mean():.1f} km")
print(f"Median pace: {df_clean['pace_min_per_km'].median():.2f} min/km")

print("\nDistance distribution by age group:")
print(
    df_clean
    .group_by("age_group")
    .agg([
        pl.col("distance").mean().alias("avg_distance"),
        pl.col("athlete").n_unique().alias("athletes")
    ])
    .sort("avg_distance", descending=True)
)

=== Quick Insights ===

Total athletes: 36,412
Average weekly distance: 38.9 km
Median pace: 5.43 min/km

Distance distribution by age group:
shape: (3, 3)
┌───────────┬──────────────┬──────────┐
│ age_group ┆ avg_distance ┆ athletes │
│ ---       ┆ ---          ┆ ---      │
│ str       ┆ f64          ┆ u32      │
╞═══════════╪══════════════╪══════════╡
│ 18 - 34   ┆ 39.135708    ┆ 12242    │
│ 35 - 54   ┆ 38.79717     ┆ 21601    │
│ 55 +      ┆ 38.101772    ┆ 2569     │
└───────────┴──────────────┴──────────┘


In [ ]:
# Save as Parquet
output_path = DATA_PROCESSED / "running_weekly_clean.parquet"
df_clean.write_parquet(output_path)

print(f"✅ Clean data saved to: {output_path}")
print(f"File size: {output_path.stat().st_size / 1_000_000:.2f} MB")

✅ Clean data saved to: /content/drive/MyDrive/virtual-running-coach/data/processed/running_weekly_clean.parquet
File size: 34.98 MB


In [ ]:
olympic_file = DATA_RAW / "athlete_events.csv"

if olympic_file.exists():
    df_olympic = pl.read_csv(
        olympic_file,
        null_values=["NA", ""],
        infer_schema_length=10000
    )

    # Keep only Athletics (running events)
    df_running_oly = (
        df_olympic
        .filter(pl.col("Sport") == "Athletics")
        .filter(pl.col("Event").str.contains("(?i)metre|marathon|relay"))
        .select(["Name", "Sex", "Age", "Team", "Year", "Event", "Medal", "City"])
    )

    print(f"Olympic running events: {df_running_oly.shape[0]:,} rows")
    print(df_running_oly.head())

    # Save
    df_running_oly.write_parquet(DATA_PROCESSED / "olympic_running.parquet")
    print("✅ Olympic running data saved")

Olympic running events: 27,346 rows
shape: (5, 8)
┌────────────────────┬─────┬─────┬─────────────┬──────┬───────────────────┬───────┬────────────────┐
│ Name               ┆ Sex ┆ Age ┆ Team        ┆ Year ┆ Event             ┆ Medal ┆ City           │
│ ---                ┆ --- ┆ --- ┆ ---         ┆ ---  ┆ ---               ┆ ---   ┆ ---            │
│ str                ┆ str ┆ i64 ┆ str         ┆ i64  ┆ str               ┆ str   ┆ str            │
╞════════════════════╪═════╪═════╪═════════════╪══════╪═══════════════════╪═══════╪════════════════╡
│ Cornelia "Cor"     ┆ F   ┆ 18  ┆ Netherlands ┆ 1932 ┆ Athletics Women's ┆ null  ┆ Los Angeles    │
│ Aalten (-Strann…   ┆     ┆     ┆             ┆      ┆ 100 metres        ┆       ┆                │
│ Cornelia "Cor"     ┆ F   ┆ 18  ┆ Netherlands ┆ 1932 ┆ Athletics Women's ┆ null  ┆ Los Angeles    │
│ Aalten (-Strann…   ┆     ┆     ┆             ┆      ┆ 4 x 100 metr…     ┆       ┆                │
│ Jamale (Djamel-)   ┆ M   ┆ 30  ┆ France